Note: had to merge the two datasets so it would stop overwritting each other

In [ ]:
import pandas as pd
df = pd.read_csv('../../data/raw/movies_metadata.csv')

# Cast and Crew 
credits = pd.read_csv('../../data/raw/credits.csv')

df['id'] = df['id'].astype(str)
credits['id'] = credits['id'].astype(str)
df = df.merge(credits, on='id', how='left')


In [ ]:
#used to check if the files above merged
print(df.columns.tolist())

In [ ]:
#Dimensions
print("Shape (rows, columns):", df.shape)

#Data types
print ("\nData types:")
print (df.dtypes)


In [ ]:
# Missing Values
print ("\nMissing values per column:")
print (df.isnull().sum())

#Missing cast and crew data
print ("\nMissing cast and crew:")
print(df[['cast', 'crew']].isnull().sum())

In [ ]:
# Duplicate rows
print("\nNumber of duplicate rows:", df.drop(columns=['cast', 'crew']).duplicated().sum())

Code is showing there are 4 rows missing for both cast and crew. This caused as break in ast.literal_eval. Changing th code to skip over those 4 NaN rows (they stay untouched) but continues to parse rows w/ str data. Handled missing data by returning a safe default, preserving all 45,476 movies in the dataset

In [ ]:
# Parse the cast and crew columns 
import ast

for col in ['cast', 'crew']:
   df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
print(df[['cast', 'crew']].iloc[0].apply(type))

#Checking NaN represntations 
print ("\nOutput of NaN representations:")
print (df[['cast', 'crew']].iloc[0].apply(type)) #should show 'list'
print(df['cast'].isnull().sum()) #should show 4


In [ ]:
# Checking for invalid or duplicated movie IDs
print("id dtype:", df['id'].dtype)
print("Missing IDs:", df['id'].isnull().sum())

print('id' in df.columns)

#See what the duplicates are
duplicate_ids = df[df['id'].duplicated(keep=False)]
print("Number of rows involved in duplicate ID's:", len(duplicate_ids))
print(duplicate_ids[['id', 'title', 'release_date']].sort_values('id'))

#Clean up of duplicated titles 
df = df.drop_duplicates(subset='id', keep='first')
print("Rows after removing duplicated ID's:", len(df))


By identifying what movies were duplicated, I can see that some movies were entered multiple times. Started with 44 duplicated titles but then jumped to 89 once adding 'director', 'cast_size' etc. The 149 rows involved tell me movies are appearing 3-4 times. Now that I determined the data is genuine and not mismatched, I can proceed with clean up by keeping just one copy of each

In [ ]:
# Extracting Director 
def get_director(crew_list):
   if not isinstance(crew_list, list):
       return None
   for person in crew_list:
       if person ['job'] == 'Director':
           return person['name']

df['director'] = df['crew'].apply(get_director)
print("\nDirector:", df['director'])

#Extracting Top five cast members
def get_top_five_cast(cast_list):
    if not isinstance(cast_list, list):
        return []
    return [person['name'] for person in cast_list[:5]]

df['top_five_cast'] = df['cast'].apply(get_top_five_cast)
print("\nTop Five Cast:", df['top_five_cast'])

#Extracting Cast Size
def get_cast_size(cast_list):
    if not isinstance(cast_list, list):
        return 0
    return len(cast_list)

df['cast_size'] = df['cast'].apply(get_cast_size)
print("\nCast Size:", df['cast_size'])

#Extracting Crew Size
def get_crew_size(crew_list):
    if not isinstance(crew_list, list):
        return 0
    return len(crew_list)

df['crew_size'] = df['crew'].apply(get_crew_size)
print("\nCrew Size:", df['crew_size'])

In [ ]:
#Storing cast members in consistent format 
def get_all_cast_names(cast_list):
    if not isinstance(cast_list, list):
        return []
    return [person['name']for person in cast_list]

df['cast_names'] = df['cast'].apply(get_all_cast_names)
print("\nAll Cast Names:", df['cast_names'])

#Checking for an inconsistencys
print(df['cast_names'].apply(type).value_counts())


Now that I have extracted the data I needed, I will be deleting 'cast' and 'crew' to start focusing on organizing my data in a consistent format. 

In [ ]:
#Dropping raw cast and crew columns     
df = df.drop(columns=['cast', 'crew'], errors='ignore')
print("\nRemaining columns after dropping cast and crew", df.columns.tolist())

In [ ]:
df.info()

In [ ]:
#confirm Duplicates have been deleted 
print("Row count (len):", len(df))
print("Row count (shape):", df.shape[0])
print("Duplicate IDs remaining:", df['id'].duplicated().sum())
